In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, event, text
from sqlalchemy.engine import URL

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'
PROCESSED_DIR = os.path.join(ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(ROOT, 'reports')
DB_PATH = os.path.join(PROCESSED_DIR, 'bluestock_dw.db')

engine = create_engine(URL.create('sqlite', database=DB_PATH))

@event.listens_for(engine, 'connect')
def turn_on_foreign_keys(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute('PRAGMA foreign_keys = ON')
    cursor.close()

In [2]:
bounds = pd.read_sql_query("""
SELECT n.amfi_code,
       MIN(d.full_date) AS first_date,
       MAX(d.full_date) AS last_date,
       COUNT(*)         AS nav_rows
FROM fact_nav AS n
JOIN dim_date AS d ON d.date_key = n.date_key
GROUP BY n.amfi_code
""", engine)

bounds['first_date'] = pd.to_datetime(bounds['first_date'])
bounds['last_date'] = pd.to_datetime(bounds['last_date'])

END_DATE = bounds['last_date'].max()

print('Funds:', len(bounds))
print('End date:', END_DATE.date())
print('Oldest first date:', bounds['first_date'].min().date(), '| newest first date:', bounds['first_date'].max().date())

stale = bounds[bounds['last_date'] < END_DATE]
print('Funds whose last NAV is before the end date:', len(stale))

Funds: 40
End date: 2026-05-29
Oldest first date: 2022-01-03 | newest first date: 2022-01-03
Funds whose last NAV is before the end date: 0
